In [ ]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("PostgresPySpark") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4") \
    .enableHiveSupport() \
    .getOrCreate()

# PostgreSQL connection
jdbc_url = "jdbc:postgresql://host.docker.internal:5432/tpc"

properties = {
    "user": "postgres",
    "password": "tiger123",
    "driver": "org.postgresql.Driver"
}

# Create schema/database
spark.sql("CREATE DATABASE IF NOT EXISTS prod_db")

# Set schema
spark.sql("USE prod_db")

tables = ["customer","lineitem","nation","orders","part","partsupp","region","supplier"]

# Read PostgreSQL tables
for t in tables:

    df = spark.read.jdbc(
        url=jdbc_url,
        table=t,
        properties=properties
    )
    try:
        df.write.mode("overwrite").saveAsTable(f"prod_db.{t}")
        print(f"Loaded table: {t}")
    # Save into Spark catalog
    except:
        print(f"spark_catalog already created")
        continue

In [ ]:
spark.sql("""select * from prod_db.customer """).show()

In [ ]:
spark.sql(""" use prod_db""").show()

In [ ]:
spark.sql("""show catalogs """).show()

In [ ]:
spark.sql(""" show  tables """).show()

In [ ]:
spark.sql(""" select * from orders
where limit 4""").show()

In [ ]:
spark.sql(""" select * from customer
where limit 4""").show()

In [ ]:
# all customer rows that have c_nationkey = 20
spark.sql(""" select * from customer
where c_nationkey = 20""").show()

In [ ]:
# all customer rows that have c_nationkey = 20 and c_acctbal > 1000

spark.sql(""" select * from customer
where c_nationkey = 20 and  c_acctbal > 1000""").show()

In [ ]:
#all customer rows that have c_nationkey = 20 or c_acctbal > 1000

spark.sql(""" select * from customer
where c_nationkey = 20 or  c_acctbal > 1000""").show()

In [ ]:
#all customer rows that have (c_nationkey = 20 and c_acctbal > 1000) or rows that have c_nationkey = 11

spark.sql(""" select * from customer
where (c_nationkey = 20 or  c_acctbal > 1000) or  c_nationkey = 11
""").show()

In [ ]:
#-- all customer rows where the name has a 381 in it

spark.sql(""" select * from customer
where c_name like  "%381%"
""").show()

In [ ]:
#-- all customer rows where the name ends with a 381


spark.sql(""" select * from customer
where c_name like  "%381"
""").show()

In [ ]:
#-- all customer rows where the name starts with a 381
spark.sql(""" select * from customer
where c_name like  "%381"
""").show()

In [ ]:
#- all customer rows where the name has a combination of any character and 9 and 1

spark.sql(""" select * from customer
where c_name like  '%9%' and c_name like  '%1%'
""").show()

In [ ]:
#all customer rows which have nationkey = 10 or nationkey = 20
spark.sql("""select * from customer
where c_nationkey = 10 or c_nationkey = 10""").show()

In [ ]:
#all customer rows which have do not have nationkey as 10 or 20
spark.sql("""select * from customer
where c_nationkey not in (10, 20)""").show()

In [ ]:
#*
spark.sql(""" select count(*) from orders""").show()

In [ ]:
spark.sql(""" select * from lineitem""").show()

In [ ]:
#Will show the first ten customer records with the lowest custkey
#-- rows are ordered in ASC order by default
spark.sql("""select c_custkey from customer 
order by c_custkey 
limit 10
""").show()



In [ ]:
#Will show the first ten customer's records with the highest custkey
spark.sql("""select c_custkey from customer 
order by c_custkey desc
limit 10
""").show()

In [ ]:
spark.sql(""" select * from lineitem
where limit 4""").show()

In [ ]:
spark.sql(""" select * from orders
where limit 4""").show()

In [ ]:
#join

spark.sql(""" select l.l_commitdate, l.l_orderkey, o.o_orderkey from orders o
inner join lineitem l on o.o_custkey = l.l_orderkey
""").show()

In [ ]:
#join

spark.sql(""" select l.l_commitdate, l.l_orderkey, o.o_orderkey from orders o
left join lineitem l on o.o_custkey = l.l_orderkey
""").show()

In [ ]:
#get the order before and after 5 days

spark.sql(""" select o.o_orderkey, l.l_orderkey from orders o
inner join lineitem l on o.o_orderkey= l.l_orderkey
where o_orderdate between  l_shipdate - interval '5' day and l_shipdate + interval '5' day
limit 5
""").show()

In [ ]:
spark.sql("""select o1.o_orderdate,o1.o_totalprice, o2.o_orderkey  from orders o1
join orders o2 on o1.o_custkey = o2.o_custkey
where year(o1.o_orderdate) = year(o2.o_orderdate) and weekofyear(o1.o_orderdate) = weekofyear(o2.o_orderdate)
""").show()

In [ ]:
spark.sql("""select count(*),  o_orderdate from orders
group by o_orderdate
limit 4
""").show()

In [ ]:
spark.sql("""select count(*),  o_orderdate from orders
group by o_orderdate
having count(*) > 60
limit 4
""").show()

In [ ]:
spark.sql(""" select count(*), o_orderstatus from orders
group by o_orderstatus
limit 4
""").show()

In [ ]:
#conditional statment 

spark.sql("""select *
,
case when o_orderstatus = "O" then 1 
      when o_orderstatus = "F" then 2
      when o_orderstatus = "P" then 3
else "not define"
end
from orders """).show()

In [ ]:
spark.sql(""" SELECT c_custkey, c_name FROM customer WHERE c_name LIKE'%_91%'
UNION
SELECT c_custkey, c_name FROM customer WHERE c_name LIKE '%_91'
limit 10""").show()

In [ ]:
spark.sql(""" select * from customer

limit 4
""").show()

In [ ]:
spark.sql(""" SELECT c_custkey, c_name FROM customer WHERE c_name LIKE'%_91%'
UNION ALL
SELECT c_custkey, c_name FROM customer WHERE c_name LIKE '%_91'
limit 10""").show()

In [ ]:
#EXCEPT will get the rows in the first query result that is not in the second query result, 0 rows
spark.sql(""" SELECT c_custkey, c_name FROM customer WHERE c_name LIKE'%_91%'
EXCEPT
SELECT c_custkey, c_name FROM customer WHERE c_name LIKE '%_91'
limit 10""").show()

In [ ]:
saprk.sql("""Datediff(date "2025-04-30", "2025-04-31" ) """).show()